# Rust 118: Redis — Practice

Companion exercises for `18_redis_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and every demonstration cell has an exercise, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the `todo!()` bodies; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. Connecting: Client and MultiplexedConnection](#1-connecting-client-and-multiplexedconnection)
- [2. Strings: GET, SET, INCR](#2-strings-get-set-incr)
- [3. Expiry: EXPIRE, TTL, PERSIST](#3-expiry-expire-ttl-persist)
- [4. Hashes: Fields Within a Key](#4-hashes-fields-within-a-key)
- [5. Lists and Sets](#5-lists-and-sets)
- [6. Pipelines and Transactions](#6-pipelines-and-transactions)
- [7. Pub/Sub](#7-pubsub)
- [8. Values with serde](#8-values-with-serde)
- [9. Cache-Aside: Chapter 17 as a Client](#9-cache-aside-chapter-17-as-a-client)
- [10. Mixed Review](#10-mixed-review)
- [11. What Comes Next](#11-what-comes-next)

§1–§9 each have exercises for the demonstration cells of the example notebook (§9.1 is the *given* origin server, so §9's exercises are 9.2 and 9.3); §10 mixes corners. §10 of the example notebook is the E-code catalogue.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from an earlier cell is visible in every cell below it — including `use` lines.
3. **Items are order-independent.** A struct or fn may be used above the cell that defines it.

> **Item names must be unique across the whole notebook.** Every cell shares one namespace; two functions with the same name are a redefinition error even in different sections. The exercise names here are unique by construction — do not introduce colliding helpers. The same rule covers `use` lines: the first cell that needs a name is the one that imports; later cells rely on persistence — a repeated `use` is E0252.

### The stub convention

Each exercise cell declares signatures with contract doc comments and a `// --- check (do not modify) ---` block of asserts. The body you must fill in is `todo!()`, which compiles but **panics** when executed. Replace it and re-run; an `Exercise N.M passed` line means you are done.

### Four notes specific to this chapter

- **Every check builds its own desk.** The given `redis_desk()` helper (§1.1) starts a fresh RESP2 server on an ephemeral port and returns its URL; the origin server (§9) is the given `origin_desk()` the same way. Nothing here touches a real network — no external service can make a check flake.
- **Async stubs are driven by the check's runtime.** A stub that needs the desk is an `async fn` taking the desk URL; the check builds a `Runtime`, `block_on`s the stub, and asserts on its return value. Sync helpers (like the given servers) run on plain threads — never mix the two by calling blocking code inside async.
- **Name the reply type at every call site.** The command's decode target is your annotation: `let v: Option<String> = con.get(k).await?` — an un-annotated command is E0282, §10.1 of the example notebook.
- **The crate cell below is given and must run first** (once per kernel session, features included). A `cannot find crate` diagnostic means it has not run — restart and go from the top.

### Expect warnings until you finish

Three warning families are normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the `use` lines serving a stub you have not solved yet;
- **stub-induced dead code** (`never used` / `never constructed`) — given helpers that only a *solution* calls.

Chapter 14's fourth family (`variable does not need to be mutable`) is designed out: no stub takes a `mut` binding parameter — connections are created *inside* the stubs, from the URL they are handed.


In [ ]:
// 0.1 The chapter's crate cell: run once per kernel session, features included.
:dep redis = { version = "0.27", default-features = false, features = ["tokio-comp", "serde"] }
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep tokio = { version = "1", features = ["full"] }
:dep futures-util = { version = "0.3", default-features = false, features = ["sink", "std"] }
:dep reqwest = { version = "0.12", default-features = false, features = ["json", "charset"] }

let rt = tokio::runtime::Runtime::new().unwrap();
let answer = rt.block_on(async { 40 + 2 });
assert_eq!(answer, 42);
println!("crates loaded; block_on smoke: {}", answer);


## 1. Connecting: Client and MultiplexedConnection

`redis::Client::open(url)` parses; `get_multiplexed_tokio_connection()` connects; `con.clone()` is a handle, not a socket; every command's reply type is declared at the call site.

**See**: §1 of the example notebook.


In [ ]:
// 1.1 Given: the local RESP2 desk — the example notebook's §0.2 verbatim.
// Every exercise below calls redis_desk() for a URL of its own.
use std::collections::HashMap;
use std::io::{Read, Write};
use std::sync::{Arc, Mutex};

fn bulk(s: &str) -> String { format!("${}\r\n{}\r\n", s.len(), s) }

fn arr(items: &[String]) -> String {
    let mut out = format!("*{}\r\n", items.len());
    // EVERY element needs its own CRLF: a count element like ":1" without one
    // leaves the frame unterminated and the client's parser waits forever.
    for i in items { out.push_str(&crlf(i.clone())); }
    out
}

fn crlf(mut s: String) -> String {
    if !s.ends_with("\r\n") { s.push_str("\r\n"); }
    s
}

fn find(hay: &[u8], from: usize, needle: &[u8]) -> Option<usize> {
    if needle.len() > hay.len() || from > hay.len() - needle.len() { return None; }
    (from..=hay.len() - needle.len()).find(|&i| &hay[i..i + needle.len()] == needle)
}

fn now_ms() -> u128 {
    std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).unwrap().as_millis()
}

#[derive(Default)]
struct Db {
    kv: HashMap<String, String>,
    expiry_ms: HashMap<String, u128>,
    lists: HashMap<String, Vec<String>>,
    sets: HashMap<String, Vec<String>>,
    hashes: HashMap<String, Vec<(String, String)>>,
    subs: HashMap<String, Vec<usize>>,
    psubs: HashMap<String, Vec<usize>>,
}

type SharedDb = Arc<Mutex<Db>>;
type SubTxMap = Arc<Mutex<HashMap<usize, Vec<std::sync::mpsc::Sender<(String, String)>>>>>;

fn alive(d: &Db, key: &str) -> bool {
    let exists = d.kv.contains_key(key) || d.lists.contains_key(key)
        || d.sets.contains_key(key) || d.hashes.contains_key(key);
    exists && d.expiry_ms.get(key).map_or(true, |&t| t > now_ms())
}

fn sweep(d: &mut Db) {
    let dead: Vec<String> = d.expiry_ms.iter()
        .filter(|e| *e.1 <= now_ms()).map(|e| e.0.clone()).collect();
    for k in dead {
        d.kv.remove(&k); d.expiry_ms.remove(&k);
        d.lists.remove(&k); d.sets.remove(&k); d.hashes.remove(&k);
    }
}

fn arg(cmd: &[Vec<u8>], i: usize) -> String {
    cmd.get(i).map(|b| String::from_utf8_lossy(b).into_owned()).unwrap_or_default()
}

fn dispatch(d: &mut Db, subs_tx: &SubTxMap, cmd: &[Vec<u8>]) -> String {
    sweep(d);
    let name = String::from_utf8_lossy(&cmd[0]).to_uppercase();
    match name.as_str() {
        "PING" => "+PONG".to_string(),
        "ECHO" => bulk(&arg(cmd, 1)),
        "SET" | "SETEX" => {
            let k = arg(cmd, 1);
            if name == "SETEX" {
                let secs: u128 = arg(cmd, 2).parse().unwrap_or(0);
                d.expiry_ms.insert(k.clone(), now_ms() + secs * 1000);
                d.kv.insert(k, arg(cmd, 3));
                return "+OK".to_string();
            }
            let mut ttl = None;
            let mut i = 3;
            while i < cmd.len() {
                let o = arg(cmd, i).to_uppercase();
                if o == "EX" { ttl = Some(now_ms() + arg(cmd, i + 1).parse::<u128>().unwrap_or(0) * 1000); i += 2; }
                else if o == "PX" { ttl = Some(now_ms() + arg(cmd, i + 1).parse::<u128>().unwrap_or(0)); i += 2; }
                else { i += 1; }
            }
            if let Some(t) = ttl { d.expiry_ms.insert(k.clone(), t); }
            d.kv.insert(k, arg(cmd, 2));
            "+OK".to_string()
        }
        "GET" => {
            let k = arg(cmd, 1);
            if alive(d, &k) { bulk(&d.kv[&k]) } else { "$-1\r\n".to_string() }
        }
        "DEL" => {
            let mut n = 0;
            for i in 1..cmd.len() {
                let k = arg(cmd, i);
                let mut gone = d.kv.remove(&k).is_some();
                gone |= d.lists.remove(&k).is_some();
                gone |= d.sets.remove(&k).is_some();
                gone |= d.hashes.remove(&k).is_some();
                d.expiry_ms.remove(&k);
                if gone { n += 1; }
            }
            format!(":{}", n)
        }
        "EXISTS" => format!(":{}", if alive(d, &arg(cmd, 1)) { 1 } else { 0 }),
        "TTL" | "PTTL" => {
            let k = arg(cmd, 1);
            if !alive(d, &k) {
                let lingering = d.kv.contains_key(&k) || d.lists.contains_key(&k)
                    || d.sets.contains_key(&k) || d.hashes.contains_key(&k);
                if lingering { ":-1".to_string() } else { ":-2".to_string() }
            } else if let Some(&t) = d.expiry_ms.get(&k) {
                let ms = t.saturating_sub(now_ms());
                if name == "PTTL" { format!(":{}", ms) } else { format!(":{}", (ms + 999) / 1000) }
            } else { ":-1".to_string() }
        }
        "EXPIRE" | "PEXPIRE" => {
            let k = arg(cmd, 1);
            let mult: u128 = if name == "EXPIRE" { 1000 } else { 1 };
            let amount: u128 = arg(cmd, 2).parse().unwrap_or(0);
            if alive(d, &k) { d.expiry_ms.insert(k, now_ms() + amount * mult); ":1".to_string() } else { ":0".to_string() }
        }
        "PERSIST" => {
            let k = arg(cmd, 1);
            if alive(d, &k) && d.expiry_ms.remove(&k).is_some() { ":1".to_string() } else { ":0".to_string() }
        }
        "INCR" | "INCRBY" => {
            let k = arg(cmd, 1);
            let delta: i64 = if name == "INCR" { 1 } else { arg(cmd, 2).parse().unwrap_or(0) };
            let v: i64 = d.kv.get(&k).and_then(|s| s.parse().ok()).unwrap_or(0);
            let nv = v + delta;
            d.kv.insert(k, nv.to_string());
            format!(":{}", nv)
        }
        "HSET" => {
            let k = arg(cmd, 1);
            let mut added = 0i64;
            let e = d.hashes.entry(k).or_default();
            let mut a = 2;
            while a + 1 < cmd.len() {
                let f = arg(cmd, a);
                let v = arg(cmd, a + 1);
                if let Some(p) = e.iter_mut().find(|(pf, _)| *pf == f) { p.1 = v; }
                else { e.push((f, v)); added += 1; }
                a += 2;
            }
            format!(":{}", added)
        }
        "HMSET" => {
            let k = arg(cmd, 1);
            let e = d.hashes.entry(k).or_default();
            let mut a = 2;
            while a + 1 < cmd.len() {
                let f = arg(cmd, a);
                let v = arg(cmd, a + 1);
                if let Some(p) = e.iter_mut().find(|(pf, _)| *pf == f) { p.1 = v; }
                else { e.push((f, v)); }
                a += 2;
            }
            "+OK".to_string()
        }
        "HGET" => {
            let k = arg(cmd, 1);
            let f = arg(cmd, 2);
            match d.hashes.get(&k).and_then(|h| h.iter().find(|(pf, _)| pf == &f)) {
                Some((_, v)) => bulk(v),
                None => "$-1\r\n".to_string(),
            }
        }
        "HGETALL" => match d.hashes.get(&arg(cmd, 1)) {
            Some(h) => {
                let mut items = Vec::new();
                for (f, v) in h { items.push(bulk(f)); items.push(bulk(v)); }
                arr(&items)
            }
            None => "*0\r\n".to_string(),
        },
        "HDEL" => {
            let k = arg(cmd, 1);
            let f = arg(cmd, 2);
            match d.hashes.get_mut(&k) {
                Some(h) => { let before = h.len(); h.retain(|(pf, _)| pf != &f); format!(":{}", (before - h.len()) as i64) }
                None => ":0".to_string(),
            }
        }
        "LPUSH" => {
            let k = arg(cmd, 1);
            let e = d.lists.entry(k).or_default();
            for i in 2..cmd.len() { e.insert(0, arg(cmd, i)); }
            format!(":{}", e.len())
        }
        "RPUSH" => {
            let k = arg(cmd, 1);
            let e = d.lists.entry(k).or_default();
            for i in 2..cmd.len() { e.push(arg(cmd, i)); }
            format!(":{}", e.len())
        }
        "LRANGE" => {
            let k = arg(cmd, 1);
            let start: i64 = arg(cmd, 2).parse().unwrap_or(0);
            let stop: i64 = arg(cmd, 3).parse().unwrap_or(-1);
            match d.lists.get(&k) {
                Some(v) => {
                    let n = v.len() as i64;
                    let s = if start < 0 { (n + start).max(0) } else { start.min(n) };
                    let e = if stop < 0 { n + stop + 1 } else { (stop + 1).min(n) };
                    let e = e.max(s);
                    arr(&v[s as usize..e as usize].iter().map(|x| bulk(x)).collect::<Vec<_>>())
                }
                None => "*0\r\n".to_string(),
            }
        }
        "LLEN" => format!(":{}", d.lists.get(&arg(cmd, 1)).map_or(0, |v| v.len())),
        "SADD" => {
            let k = arg(cmd, 1);
            let mut added = 0;
            let e = d.sets.entry(k).or_default();
            for i in 2..cmd.len() {
                let v = arg(cmd, i);
                if !e.contains(&v) { e.push(v); added += 1; }
            }
            format!(":{}", added)
        }
        "SREM" => {
            let k = arg(cmd, 1);
            let mut removed = 0;
            if let Some(e) = d.sets.get_mut(&k) {
                for i in 2..cmd.len() {
                    let v = arg(cmd, i);
                    if let Some(pos) = e.iter().position(|x| *x == v) { e.remove(pos); removed += 1; }
                }
            }
            format!(":{}", removed)
        }
        "SMEMBERS" => {
            let items: Vec<String> = d.sets.get(&arg(cmd, 1))
                .map(|v| v.iter().map(|x| bulk(x)).collect()).unwrap_or_default();
            arr(&items)
        }
        "SCARD" => format!(":{}", d.sets.get(&arg(cmd, 1)).map_or(0, |v| v.len())),
        "SISMEMBER" => {
            let has = d.sets.get(&arg(cmd, 1)).map_or(false, |s| s.contains(&arg(cmd, 2)));
            format!(":{}", if has { 1 } else { 0 })
        }
        "PUBLISH" => {
            let ch = arg(cmd, 1);
            let payload = arg(cmd, 2);
            let mut count = 0;
            let mut targets: Vec<usize> = d.subs.get(&ch).cloned().unwrap_or_default();
            for (pat, ids) in d.psubs.iter() {
                if pat.strip_suffix('*').map_or(false, |p| ch.starts_with(p)) {
                    targets.extend(ids.iter().copied());
                }
            }
            let st = subs_tx.lock().unwrap();
            for id in targets {
                if let Some(txs) = st.get(&id) {
                    for t in txs { t.send((ch.clone(), payload.clone())).ok(); count += 1; }
                }
            }
            format!(":{}", count)
        }
        "CLIENT" | "SELECT" | "AUTH" => "+OK".to_string(),
        "HELLO" => "-ERR unknown command 'HELLO'".to_string(),
        "CONFIG" | "COMMAND" => "*0\r\n".to_string(),
        _ => format!("-ERR unknown command '{}'", name),
    }
}

enum Parsed { None, Skip(#[allow(dead_code)] usize), Cmd(Vec<Vec<u8>>, usize) }

fn parse_one(buf: &[u8]) -> Parsed {
    if buf.is_empty() { return Parsed::None; }
    if buf[0] == b'*' {
        let hdr = match find(buf, 0, b"\r\n") { Some(i) => i, None => return Parsed::None };
        let n: usize = match std::str::from_utf8(&buf[1..hdr]).ok().and_then(|s| s.parse().ok()) {
            Some(n) => n, None => return Parsed::Skip(hdr + 2),
        };
        let mut p = hdr + 2;
        let mut args = Vec::new();
        for _ in 0..n {
            if p >= buf.len() { return Parsed::None; }
            if buf[p] != b'$' { return Parsed::Skip(p); }
            let len_end = match find(buf, p, b"\r\n") { Some(i) => i, None => return Parsed::None };
            let len: usize = match std::str::from_utf8(&buf[p + 1..len_end]).ok().and_then(|s| s.parse().ok()) {
                Some(l) => l, None => return Parsed::Skip(len_end + 2),
            };
            p = len_end + 2;
            if p + len + 2 > buf.len() { return Parsed::None; }
            args.push(buf[p..p + len].to_vec());
            p += len + 2;
        }
        Parsed::Cmd(args, p)
    } else {
        let line_end = match find(buf, 0, b"\n") { Some(i) => i, None => return Parsed::None };
        let line = String::from_utf8_lossy(&buf[..line_end]).trim().to_string();
        if line.is_empty() { return Parsed::Skip(line_end + 1); }
        Parsed::Cmd(line.split_whitespace().map(|s| s.as_bytes().to_vec()).collect(), line_end + 1)
    }
}

/// Start a fresh RESP2 desk; return its URL. Given.
fn redis_desk() -> String {
    let listener = std::net::TcpListener::bind("127.0.0.1:0").unwrap();
    let port = listener.local_addr().unwrap().port();
    let db: SharedDb = Arc::new(Mutex::new(Db::default()));
    let subs_tx: SubTxMap = Arc::new(Mutex::new(HashMap::new()));
    let next_id = Arc::new(Mutex::new(0usize));
    std::thread::spawn(move || {
        for stream in listener.incoming().flatten() {
            let db = Arc::clone(&db);
            let subs_tx = Arc::clone(&subs_tx);
            let next_id = Arc::clone(&next_id);
            std::thread::spawn(move || {
                let mut r = stream.try_clone().unwrap();
                let mut w = stream;
                r.set_read_timeout(Some(std::time::Duration::from_millis(10))).ok();
                let mut my_id: Option<usize> = None;
                let (tx, rx) = std::sync::mpsc::channel::<(String, String)>();
                let mut buf: Vec<u8> = Vec::new();
                let mut tmp = [0u8; 8192];
                let mut queue: Option<Vec<Vec<Vec<u8>>>> = None;
                loop {
                    loop {
                        match rx.try_recv() {
                            Ok((ch, payload)) => {
                                let frame = arr(&[bulk("message"), bulk(&ch), bulk(&payload)]);
                                w.write_all(frame.as_bytes()).ok();
                                w.flush().ok();
                            }
                            Err(_) => break,
                        }
                    }
                    match r.read(&mut tmp) {
                        Ok(0) => break,
                        Ok(n) => buf.extend_from_slice(&tmp[..n]),
                        Err(ref e) if e.kind() == std::io::ErrorKind::WouldBlock
                            || e.kind() == std::io::ErrorKind::TimedOut => continue,
                        Err(_) => break,
                    }
                    while let Parsed::Cmd(cmd, used) = parse_one(&buf) {
                        buf.drain(..used);
                        if cmd.is_empty() { continue; }
                        let name = String::from_utf8_lossy(&cmd[0]).to_uppercase();
                        if name == "SUBSCRIBE" || name == "PSUBSCRIBE" {
                            if my_id.is_none() {
                                let mut g = next_id.lock().unwrap();
                                my_id = Some(*g);
                                *g += 1;
                            }
                            let id = my_id.unwrap();
                            let mut names = Vec::new();
                            {
                                let mut st = subs_tx.lock().unwrap();
                                let mut d = db.lock().unwrap();
                                for a in 1..cmd.len() {
                                    let ch = arg(&cmd, a);
                                    names.push(ch.clone());
                                    st.entry(id).or_default().push(tx.clone());
                                    if name == "SUBSCRIBE" { d.subs.entry(ch).or_default().push(id); }
                                    else { d.psubs.entry(ch).or_default().push(id); }
                                }
                            }
                            let mut confirm = String::new();
                            for (i, ch) in names.iter().enumerate() {
                                confirm.push_str(&arr(&[bulk(&name.to_lowercase()), bulk(ch), format!(":{}", i + 1)]));
                            }
                            w.write_all(confirm.as_bytes()).ok();
                            w.flush().ok();
                            continue;
                        }
                        if name == "UNSUBSCRIBE" || name == "PUNSUBSCRIBE" {
                            let kind = if name == "UNSUBSCRIBE" { "unsubscribe" } else { "punsubscribe" };
                            let frame = arr(&[bulk(kind), bulk(&arg(&cmd, 1)), ":0".to_string()]);
                            w.write_all(frame.as_bytes()).ok();
                            w.flush().ok();
                            continue;
                        }
                        if name == "PING" && my_id.is_some() {
                            let frame = arr(&[bulk("pong"), bulk("")]);
                            w.write_all(frame.as_bytes()).ok();
                            w.flush().ok();
                            continue;
                        }
                        let reply = match name.as_str() {
                            "MULTI" => {
                                if queue.is_none() { queue = Some(Vec::new()); "+OK".to_string() }
                                else { "-ERR MULTI calls can not be nested".to_string() }
                            }
                            "EXEC" => match queue.take() {
                                None => "-ERR EXEC without MULTI".to_string(),
                                Some(q) => {
                                    let mut out = format!("*{}\r\n", q.len());
                                    for c in q {
                                        let mut d = db.lock().unwrap();
                                        out.push_str(&crlf(dispatch(&mut d, &subs_tx, &c)));
                                    }
                                    out
                                }
                            },
                            "DISCARD" => { queue = None; "+OK".to_string() }
                            _ => {
                                if queue.is_some() {
                                    queue.as_mut().unwrap().push(cmd);
                                    "+QUEUED".to_string()
                                } else {
                                    let mut d = db.lock().unwrap();
                                    dispatch(&mut d, &subs_tx, &cmd)
                                }
                            }
                        };
                        w.write_all(crlf(reply).as_bytes()).ok();
                        w.flush().ok();
                    }
                }
            });
        }
    });
    format!("redis://127.0.0.1:{}", port)
}

/// Connect to `url` and answer a PING as a String. The whole sequence —
/// Client::open, get_multiplexed_tokio_connection, cmd("PING") with its
/// reply type — is the exercise.
async fn connect_and_ping(url: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
let pong = rt.block_on(connect_and_ping(&url));
assert_eq!(pong, "PONG");
println!("Exercise 1.1 passed");


## 2. Strings: GET, SET, INCR

The string type is bytes under a key; absence reads back as `Ok(None)`; `incr` is the server-side atomic read-add-store.

**See**: §2 of the example notebook.


In [ ]:
// 2.1 set_quote stores; get_quote reads absence as None; bump_order counts.
use redis::AsyncCommands;

/// SET `key` = `value`. The reply is +OK mapped to () — the cast IS the type.
async fn set_quote(con: &mut redis::aio::MultiplexedConnection, key: &str, value: &str) {
    todo!()
}

/// GET `key` as Option<String>: None when the key is absent (not an error).
async fn get_quote(con: &mut redis::aio::MultiplexedConnection, key: &str) -> Option<String> {
    todo!()
}

/// INCRBY `key` by `delta`; return the new value. One command, no race.
async fn bump_order(con: &mut redis::aio::MultiplexedConnection, key: &str, delta: i64) -> i64 {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    set_quote(&mut con, "px:AAA", "10.0").await;
    set_quote(&mut con, "px:BBB", "20.5").await;
    assert_eq!(get_quote(&mut con, "px:AAA").await.as_deref(), Some("10.0"));
    assert_eq!(get_quote(&mut con, "px:GHOST").await, None);
    assert_eq!(bump_order(&mut con, "desk:orders", 1).await, 1);
    assert_eq!(bump_order(&mut con, "desk:orders", 4).await, 5);
});
println!("Exercise 2.1 passed");


## 3. Expiry: EXPIRE, TTL, PERSIST

Three TTL answers: `-2` gone, `-1` no countdown, `n` seconds left. Arming a missing key is `false`; `set_ex` fuses write-and-arm; expiry deletes server-side.

**See**: §3 of the example notebook.


In [ ]:
// 3.1 arm, read_back, and fuse — the session-TTL toolkit.

/// Arm a countdown of `secs` on `key`; return whether the key existed.
async fn arm(con: &mut redis::aio::MultiplexedConnection, key: &str, secs: i64) -> bool {
    todo!()
}

/// Return the TTL of `key`: -2 gone, -1 no countdown, n seconds remaining.
async fn read_back(con: &mut redis::aio::MultiplexedConnection, key: &str) -> i64 {
    todo!()
}

/// SET `key` = `value` with a TTL of `secs` in ONE command (the fused verb).
async fn fuse(con: &mut redis::aio::MultiplexedConnection, key: &str, value: &str, secs: u64) {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    let _: () = con.set("sess:u1", "ana").await.unwrap();
    assert_eq!(arm(&mut con, "sess:u1", 120).await, true);
    assert_eq!(arm(&mut con, "nope", 60).await, false);
    let t = read_back(&mut con, "sess:u1").await;
    assert!((100..=120).contains(&t), "countdown, got {}", t);
    assert_eq!(read_back(&mut con, "nope").await, -2);
    let _: () = con.set("forever", "x").await.unwrap();
    assert_eq!(read_back(&mut con, "forever").await, -1);
    fuse(&mut con, "sess:u2", "bob", 120).await;
    let t2 = read_back(&mut con, "sess:u2").await;
    assert!((100..=120).contains(&t2), "fused write armed the TTL, got {}", t2);
});
println!("Exercise 3.1 passed");


## 4. Hashes: Fields Within a Key

One key, many fields; `hset` counts genuinely-new fields; `hgetall` decodes into `HashMap<String, String>`.

**See**: §4 of the example notebook.


In [ ]:
// 4.1 The position book as one hash per user.

/// Set `field` = `value` inside hash `key`; return how many NEW fields that created.
async fn set_field(con: &mut redis::aio::MultiplexedConnection, key: &str, field: &str, value: &str) -> i64 {
    todo!()
}

/// Read one field as Option<String>; a missing field (or key) is None.
async fn get_field(con: &mut redis::aio::MultiplexedConnection, key: &str, field: &str) -> Option<String> {
    todo!()
}

/// Fetch the whole hash as a HashMap<String, String>.
async fn whole_book(con: &mut redis::aio::MultiplexedConnection, key: &str) -> std::collections::HashMap<String, String> {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    assert_eq!(set_field(&mut con, "pos:u9", "sym", "AAA").await, 1);
    assert_eq!(set_field(&mut con, "pos:u9", "qty", "100").await, 1);
    assert_eq!(set_field(&mut con, "pos:u9", "qty", "150").await, 0, "overwrite adds nothing");
    assert_eq!(get_field(&mut con, "pos:u9", "qty").await.as_deref(), Some("150"));
    assert_eq!(get_field(&mut con, "pos:u9", "px").await, None);
    let book = whole_book(&mut con, "pos:u9").await;
    assert_eq!(book.len(), 2);
    assert_eq!(book["sym"], "AAA");
});
println!("Exercise 4.1 passed");


## 5. Lists and Sets

Lists: ordered, cheap ends, `lrange` with negatives. Sets: unique members, `sadd` counts genuinely-new, sort `smembers` before comparing.

**See**: §5 of the example notebook.


In [ ]:
// 5.1 The job queue (list) and the watchlist (set).

/// Append `job` at the TAIL of list `key`; return the new length.
async fn enqueue(con: &mut redis::aio::MultiplexedConnection, key: &str, job: &str) -> i64 {
    todo!()
}

/// Insert `job` at the HEAD of list `key`; return the new length.
async fn jump_queue(con: &mut redis::aio::MultiplexedConnection, key: &str, job: &str) -> i64 {
    todo!()
}

/// Slice `key` from start..=stop (negatives welcome) as Vec<String>.
async fn window(con: &mut redis::aio::MultiplexedConnection, key: &str, start: i64, stop: i64) -> Vec<String> {
    todo!()
}

/// Add `sym` to set `key`; return 1 if it is genuinely new, 0 if already there.
async fn watch(con: &mut redis::aio::MultiplexedConnection, key: &str, sym: &str) -> i64 {
    todo!()
}

/// Return every member of set `key`, SORTED, so callers can compare.
async fn watchlist(con: &mut redis::aio::MultiplexedConnection, key: &str) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    assert_eq!(enqueue(&mut con, "desk:jobs", "audit").await, 1);
    assert_eq!(enqueue(&mut con, "desk:jobs", "rebalance").await, 2);
    assert_eq!(jump_queue(&mut con, "desk:jobs", "urgent").await, 3);
    assert_eq!(window(&mut con, "desk:jobs", 0, -1).await,
        vec!["urgent", "audit", "rebalance"]);
    assert_eq!(window(&mut con, "desk:jobs", 0, 0).await, vec!["urgent"]);
    assert_eq!(watch(&mut con, "watch:u9", "AAA").await, 1);
    assert_eq!(watch(&mut con, "watch:u9", "BBB").await, 1);
    assert_eq!(watch(&mut con, "watch:u9", "AAA").await, 0, "duplicate adds nothing");
    assert_eq!(watchlist(&mut con, "watch:u9").await, vec!["AAA", "BBB"]);
});
println!("Exercise 5.1 passed");


## 6. Pipelines and Transactions

A pipeline batches round-trips; `.ignore()` removes a reply from the tuple; `.atomic()` adds MULTI/EXEC — no interleaving.

**See**: §6 of the example notebook.


In [ ]:
// 6.1 Write two keys and read them back in ONE round trip.

/// In one pipeline: SET `k1`=`v1` and SET `k2`=`v2` (both ignored),
/// then GET `k1` and GET `k2`. Return both reads as (String, String).
async fn write_and_read(con: &mut redis::aio::MultiplexedConnection, k1: &str, v1: &str, k2: &str, v2: &str) -> (String, String) {
    todo!()
}

/// In one ATOMIC pipeline: INCRBY `key` by `delta`, then GET `key`.
/// Return the GET. No other connection's commands can interleave.
async fn counted_step(con: &mut redis::aio::MultiplexedConnection, key: &str, delta: i64) -> String {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    let (a, b) = write_and_read(&mut con, "t:a", "1", "t:b", "2").await;
    assert_eq!((a.as_str(), b.as_str()), ("1", "2"));
    let _: () = con.set("t:n", "10").await.unwrap();
    assert_eq!(counted_step(&mut con, "t:n", 5).await, "15");
    assert_eq!(counted_step(&mut con, "t:n", 5).await, "20");
});
println!("Exercise 6.1 passed");


## 7. Pub/Sub

`PUBLISH` returns the receiver count; subscribers ride a dedicated socket; `into_on_message()` consumes into a `Stream<Msg>`.

**See**: §7 of the example notebook.


In [ ]:
// 7.1 The alert tap: subscribe on one connection, publish on another.
use futures_util::StreamExt;

/// Publish `payload` on `channel` via the multiplexed connection; return the
/// number of subscribers that received it. (redis::cmd("PUBLISH") is the form.)
async fn broadcast(con: &mut redis::aio::MultiplexedConnection, channel: &str, payload: &str) -> i64 {
    todo!()
}

/// Subscribe to `channel` on a FRESH pub/sub connection (get_async_pubsub on
/// the client — not the multiplexed one), consume it with into_on_message,
/// and await the first message (with a 3s timeout). Return (channel, payload).
async fn next_alert(client: &redis::Client, channel: &str) -> (String, String) {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    // no subscribers yet: fire-and-forget
    assert_eq!(broadcast(&mut con, "fills", "AAA@10").await, 0);

    // subscribe first, give the desk a beat, then publish: count must be 1
    let tap_client = client.clone();
    let tap = tokio::spawn(async move { next_alert(&tap_client, "fills").await });
    std::thread::sleep(std::time::Duration::from_millis(200));
    assert_eq!(broadcast(&mut con, "fills", "BBB@20.5").await, 1);
    let (ch, payload) = tokio::time::timeout(std::time::Duration::from_secs(5), tap)
        .await
        .expect("tap finished")
        .expect("join ok");
    assert_eq!(ch, "fills");
    assert_eq!(payload, "BBB@20.5");
});
println!("Exercise 7.1 passed");


## 8. Values with serde

Redis stores bytes; a derived struct serialized with `serde_json` is the convention that makes them typed.

**See**: §8 of the example notebook.


In [ ]:
// 8.1 Store a Quote, get a Quote — the JSON-string convention.

#[derive(serde::Serialize, serde::Deserialize, Debug, PartialEq, Clone)]
pub struct Quote {
    pub sym: String,
    pub px: f64,
    pub seq: u64,
}

/// Serialize `quote` with serde_json and SET it under `key`.
async fn save_quote(con: &mut redis::aio::MultiplexedConnection, key: &str, quote: &Quote) {
    todo!()
}

/// GET `key` and decode it into a Quote. Absent key -> None;
/// present key -> Some(decoded). (Two Options: the crate's and serde's.)
async fn load_quote(con: &mut redis::aio::MultiplexedConnection, key: &str) -> Option<Quote> {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    let q = Quote { sym: "SPIRE".into(), px: 101.5, seq: 7 };
    save_quote(&mut con, "json:q:SPIRE", &q).await;
    assert_eq!(load_quote(&mut con, "json:q:SPIRE").await, Some(q));
    assert_eq!(load_quote(&mut con, "json:q:NOPE").await, None);
});
println!("Exercise 8.1 passed");


## 9. Cache-Aside: Chapter 17 as a Client

GET the cache; on a miss fetch the origin and `set_ex` with a freshness budget; on a hit decode and return. §9.1's origin server is given — the exercises are the pattern itself.

**See**: §9 of the example notebook.


In [ ]:
// 9.1 Given: the origin server — chapter 17's §0.2 pattern, four routes' worth.
fn origin_desk() -> String {
    let listener = std::net::TcpListener::bind("127.0.0.1:0").unwrap();
    let port = listener.local_addr().unwrap().port();
    std::thread::spawn(move || {
        for stream in listener.incoming().flatten() {
            let mut s = stream;
            let mut reader = std::io::BufReader::new(match s.try_clone() { Ok(r) => r, Err(_) => return });
            let mut line = String::new();
            use std::io::BufRead;
            if reader.read_line(&mut line).is_err() { return; }
            let path = line.split_whitespace().nth(1).unwrap_or("/").to_string();
            loop {
                let mut h = String::new();
                if reader.read_line(&mut h).is_err() || h.trim().is_empty() { break; }
            }
            let sym = path.trim_start_matches("/quote/").to_uppercase();
            let body = format!("{{\"sym\":\"{}\",\"px\":{}.5,\"seq\":1}}", sym,
                if sym == "SPIRE" { 101 } else { 55 });
            let head = format!(
                "HTTP/1.1 200 OK\r\nContent-Type: application/json\r\nContent-Length: {}\r\nConnection: close\r\n\r\n",
                body.len());
            s.write_all(head.as_bytes()).ok();
            s.write_all(body.as_bytes()).ok();
        }
    });
    format!("http://127.0.0.1:{}", port)
}

// --- check (do not modify) ---
let origin = origin_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
let body = rt.block_on(async {
    reqwest::Client::new().get(format!("{}/quote/SPIRE", origin))
        .send().await.unwrap().text().await.unwrap()
});
assert!(body.contains("\"sym\":\"SPIRE\""));
println!("Exercise 9.1 passed (given origin answers)");


In [ ]:
// 9.2 The pattern: miss -> origin -> SET EX; hit -> decode.

/// Cache-aside for `sym`. GET cache key `cache:quote:<sym>`; on a hit decode
/// and return (quote, true). On a miss: GET <origin>/quote/<sym> with the http
/// client, decode, `set_ex` the JSON under the cache key with `ttl_secs`,
/// return (quote, false).
async fn cached_quote(con: &mut redis::aio::MultiplexedConnection, http: &reqwest::Client, origin: &str, sym: &str, ttl_secs: u64) -> (Quote, bool) {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let origin = origin_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    let http = reqwest::Client::new();
    let (first, hit1) = cached_quote(&mut con, &http, &origin, "SPIRE", 60).await;
    assert!(!hit1, "first read misses");
    assert_eq!(first.sym, "SPIRE");
    let (second, hit2) = cached_quote(&mut con, &http, &origin, "SPIRE", 60).await;
    assert!(hit2, "second read hits");
    assert_eq!(first, second);
    let t: i64 = con.ttl("cache:quote:SPIRE").await.unwrap();
    assert!((55..=60).contains(&t), "the miss armed the TTL, got {}", t);
    let (other, _) = cached_quote(&mut con, &http, &origin, "QUARK", 60).await;
    assert_eq!(other.px, 55.5);
});
println!("Exercise 9.2 passed");


In [ ]:
// 9.3 The expiry end: a short budget self-destructs.

/// SET `key` = `value` with a `millis`-millisecond budget (the pexpire-precision
/// fused form is fine: set_ex rounds up to seconds, so arm it yourself with
/// set + pexpire), then sleep-free: return the TTL-milliseconds reading
/// (PTTL) so the caller can watch it burn.
async fn brief_note(con: &mut redis::aio::MultiplexedConnection, key: &str, value: &str, millis: i64) -> i64 {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    let pttl = brief_note(&mut con, "brief:note", "gone soon", 250).await;
    assert!(pttl > 0 && pttl <= 250, "budget burning, got {}", pttl);
    std::thread::sleep(std::time::Duration::from_millis(400));
    let after: Option<String> = con.get("brief:note").await.unwrap();
    assert!(after.is_none(), "expired server-side");
});
println!("Exercise 9.3 passed");


## 10. Mixed Review

Four problems, each from a different corner of the chapter.

**See**: §2 and §6 of the example notebook (10.1), §3 and §4 (10.2), §7 (10.3), §5 and §8 (10.4).


In [ ]:
// 10.1 Pipeline + counter: one round trip, three commands.

/// In ONE plain (non-atomic) pipeline: INCRBY `key` by `delta` (kept),
/// SET `key:done` = "1" (ignored), GET `key` (kept). Return the GET value.
async fn pipeline_step(con: &mut redis::aio::MultiplexedConnection, key: &str, delta: i64) -> String {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    assert_eq!(pipeline_step(&mut con, "mx:hits", 3).await, "3");
    assert_eq!(pipeline_step(&mut con, "mx:hits", 4).await, "7");
    let done: Option<String> = con.get("mx:hits:done").await.unwrap();
    assert_eq!(done.as_deref(), Some("1"), "the ignored SET still ran");
});
println!("Exercise 10.1 passed");


In [ ]:
// 10.2 Hash + TTL: a session record that cleans up after itself.

/// Build hash `key` with fields name=`name`, level=`level`, then arm a TTL of
/// `secs`. Return the hset pair: (fields_created_total, ttl_armed).
async fn session_with_ttl(con: &mut redis::aio::MultiplexedConnection, key: &str, name: &str, level: &str, secs: i64) -> (i64, bool) {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    let (created, armed) = session_with_ttl(&mut con, "mx:sess:u3", "cy", "9000", 120).await;
    assert_eq!((created, armed), (2, true));
    let t: i64 = con.ttl("mx:sess:u3").await.unwrap();
    assert!((100..=120).contains(&t), "hash TTL armed, got {}", t);
    let level: Option<String> = con.hget("mx:sess:u3", "level").await.unwrap();
    assert_eq!(level.as_deref(), Some("9000"));
});
println!("Exercise 10.2 passed");


In [ ]:
// 10.3 Pub/sub with a pattern: one subscriber, two channels, one tap.

/// PSUBSCRIBE `pattern` on a fresh pub/sub connection (the desk supports a
/// trailing-`*` pattern), consume with into_on_message, and collect the next
/// `n` messages (3s timeout each). Return (channel, payload) pairs in order.
async fn tap_pattern(client: &redis::Client, pattern: &str, n: usize) -> Vec<(String, String)> {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    let tap_client = client.clone();
    let tap = tokio::spawn(async move { tap_pattern(&tap_client, "alerts:*", 2).await });
    std::thread::sleep(std::time::Duration::from_millis(200));
    let n1 = broadcast(&mut con, "alerts:px", "SPIRE>100").await;
    let n2 = broadcast(&mut con, "alerts:qty", "AAA*1000").await;
    assert_eq!((n1, n2), (1, 1), "one pattern subscriber gets both");
    let got = tokio::time::timeout(std::time::Duration::from_secs(6), tap)
        .await.expect("tap finished").expect("join ok");
    assert_eq!(got, vec![
        ("alerts:px".to_string(), "SPIRE>100".to_string()),
        ("alerts:qty".to_string(), "AAA*1000".to_string()),
    ]);
});
println!("Exercise 10.3 passed");


In [ ]:
// 10.4 Set + serde: a watchlist of typed quotes, capped at three members.

/// Add `sym` to set `key`; if the set now holds MORE than `cap` members,
/// remove the smallest member (sorted order) to get back to `cap`.
/// Return the members after trimming, SORTED.
async fn capped_watch(con: &mut redis::aio::MultiplexedConnection, key: &str, sym: &str, cap: usize) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let url = redis_desk();
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let client = redis::Client::open(url.as_str()).unwrap();
    let mut con = client.get_multiplexed_tokio_connection().await.unwrap();
    assert_eq!(capped_watch(&mut con, "mx:cap", "CCC", 3).await, vec!["CCC"]);
    assert_eq!(capped_watch(&mut con, "mx:cap", "AAA", 3).await, vec!["AAA", "CCC"]);
    assert_eq!(capped_watch(&mut con, "mx:cap", "BBB", 3).await, vec!["AAA", "BBB", "CCC"]);
    assert_eq!(capped_watch(&mut con, "mx:cap", "DDD", 3).await, vec!["BBB", "CCC", "DDD"],
        "smallest (AAA) was trimmed");
    assert_eq!(capped_watch(&mut con, "mx:cap", "BBB", 3).await, vec!["BBB", "CCC", "DDD"],
        "duplicate adds nothing, no trim");
});
println!("Exercise 10.4 passed");


## 11. What Comes Next

| Next | What it borrows from here |
|---|---|
| **19 — PostgreSQL** | `sqlx` pools are the multiplexed connection made relational; the typed-command discipline and §6's transactions survive `await` points verbatim. |
| **20 — PyO3** | `redis-py` returns strings you parse; the typed-command + boundary-struct discipline is what the extension exports. |
| **21 — Capstone Prep** | §9's cache in front of chapter 17's feed plus chapter 19's storage is the market-data service the capstone assembles; §7's fan-out becomes its event bus. |

Carry forward: **the reply type is yours to declare; absence is a value; expiry is the invalidation protocol; the pattern is GET, miss→origin→SET EX, hit→decode.**
